# Claims Investigation Agent — Eval-Driven Development

**Workshop flow**
1. Setup — install dependencies, set API keys, clone repo
2. Build & run the agent — see the trace in LangSmith
3. Spot the problem — agent hallucinates in its rationale
4. Apply the grounding evaluator — catch the failure systematically
5. Fix the prompt — one line change
6. Re-run & compare — grounding score improves

## 1 · Setup

In [ ]:
import sys, os

# Add repo root to sys.path so 'data' and 'evals' are importable
repo_root = os.path.abspath(os.path.join(os.path.dirname("__file__"), ".."))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

assert os.path.exists(os.path.join(repo_root, "data/sources/policy_docs.md")), \
    f"Expected to find data/sources/policy_docs.md under {repo_root}"
print("Repo root:", repo_root)

# Run once if packages are missing. Skip if you installed via: uv sync
# !pip install -q langchain-openai langgraph langsmith openevals python-dotenv

In [ ]:
# Reads OPENAI_API_KEY and LANGSMITH_API_KEY from .env at the repo root.
from dotenv import load_dotenv
load_dotenv()

import os
os.environ.setdefault('LANGSMITH_TRACING', 'true')
os.environ.setdefault('LANGSMITH_PROJECT', 'claims-workshop')
print('OPENAI_API_KEY set:   ', bool(os.getenv('OPENAI_API_KEY')))
print('LANGSMITH_API_KEY set:', bool(os.getenv('LANGSMITH_API_KEY')))

## 2 · Build the agent

Four tools — one per data source. The agent decides which ones to call based on the claim.

In [ ]:
from langchain_core.tools import tool
from data.loaders import load_policy_docs, load_claims_history, load_weather_data, load_repair_estimate

@tool
def search_policy_docs() -> str:
    """Retrieve insurance policy clauses, coverage conditions, thresholds, and exclusions.
    Always call this first."""
    return load_policy_docs()

@tool
def query_claims_history(claimant_id: str) -> list:
    """Retrieve prior claims history for a claimant ID.
    Always call this to check for repeat claims."""
    return load_claims_history(claimant_id)

@tool
def query_weather_data(incident_date: str, location: str) -> dict:
    """Retrieve historical weather for an incident date (YYYY-MM-DD) and city.
    Call when the cause could be weather-related. Clause 2 thresholds: 40mm or 90 km/h."""
    return load_weather_data(incident_date, location)

@tool
def retrieve_repair_estimate(claim_id: str) -> dict:
    """Retrieve contractor repair estimate. Required by Clause 6 for claims over €10,000."""
    return load_repair_estimate(claim_id)

tools = [search_policy_docs, query_claims_history, query_weather_data, retrieve_repair_estimate]
print("Tools:", [t.name for t in tools])

In [ ]:
from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent

PROMPT_BEFORE = """You are a claims investigation assistant for EuroShield Insurance Group.

You have access to four tools:
- search_policy_docs: retrieves the insurance policy
- query_claims_history: retrieves prior claims for a claimant
- query_weather_data: retrieves historical weather for a date and location
- retrieve_repair_estimate: retrieves contractor repair estimate for a claim

Investigate the claim and provide:
- coverage_decision (covered / partial / excluded)
- settlement_recommendation (auto_settle / assign_adjuster / flag_for_investigation)
- confidence (high / medium / low)
- rationale explaining your decision"""

llm          = ChatOpenAI(model="gpt-4o", temperature=0)
agent_before = create_agent(llm, tools, system_prompt=PROMPT_BEFORE)
print("Agent ready")

## 3 · Run the agent

Two claims — one clean baseline, one with fraud signals and ambiguous evidence.

In [ ]:
def run(claim: dict, agent) -> str:
    """Run the agent and return its final response."""
    claim_text = "\n".join(f"{k}: {v}" for k, v in claim.items())
    state = agent.invoke({"messages": [HumanMessage(content=claim_text)]})
    return state["messages"][-1].content

In [ ]:
# Claim 1 — clean baseline (CLM003, Utrecht, €9,800, no prior history)
from evals.dataset import CLAIM_INPUTS
clean_claim = CLAIM_INPUTS[3]  # CLAIM-2022-0441

print("Input:", clean_claim["claim_id"], "|", clean_claim["reported_cause"])
print()
response_clean = run(clean_claim, agent_before)
print(response_clean)

In [ ]:
# Claim 2 — fraud signals (CLM007, Amsterdam, €28,900, weather data contradicts reported cause)
fraud_claim = CLAIM_INPUTS[0]  # CLAIM-2024-0891

print("Input:", fraud_claim["claim_id"], "|", fraud_claim["reported_cause"])
print()
response_fraud = run(fraud_claim, agent_before)
print(response_fraud)

## 4 · Apply the grounding evaluator

Is the rationale supported by what the tools actually returned?

We ask an LLM judge to compare the agent's response against the tool outputs in the trace.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

GROUNDING_PROMPT = """You are evaluating whether an AI agent's response is grounded
in the evidence it actually retrieved from its tools.

## Tool outputs retrieved during investigation:
{retrieved_content}

## Agent's final response:
{response}

Is the response grounded in the tool outputs above?
- GROUNDED: every factual claim traces to specific retrieved content.
- PARTIALLY_GROUNDED: mostly supported, but one claim is vague or not traceable.
- HALLUCINATED: the response asserts facts not present in the retrieved content.

Reply with one of: GROUNDED, PARTIALLY_GROUNDED, HALLUCINATED
Then one sentence explaining which claim is unsupported (if any)."""

_judge_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

def call_judge(retrieved_content: str, response: str) -> dict:
    prompt = GROUNDING_PROMPT.format(retrieved_content=retrieved_content, response=response)
    result = _judge_llm.invoke([HumanMessage(content=prompt)])
    text = result.content.strip()
    first_line = text.split("\n")[0].upper()
    comment = text.split("\n", 1)[1].strip() if "\n" in text else ""
    return {"score": first_line, "comment": comment}

print("Judge ready")

In [ ]:
def get_tool_outputs(claim: dict, agent):
    """Run the agent once and return (retrieved_content, response)."""
    claim_text = "\n".join(f"{k}: {v}" for k, v in claim.items())
    state = agent.invoke({"messages": [HumanMessage(content=claim_text)]})
    tool_outputs = {}
    for msg in state["messages"]:
        if getattr(msg, "type", None) == "tool":
            tool_outputs[msg.name] = str(msg.content)
    retrieved = "\n\n".join(f"### {k}\n{v}" for k, v in tool_outputs.items())
    response  = state["messages"][-1].content
    return retrieved, response


def _score(verdict: str) -> float:
    if "GROUNDED" in verdict and "PARTIAL" not in verdict and "HALL" not in verdict:
        return 1.0
    return 0.5 if "PARTIAL" in verdict else 0.0


def show_result(result: dict, response: str) -> None:
    score = _score(result["score"])
    label = {1.0: "✅ GROUNDED", 0.5: "⚠️  PARTIAL", 0.0: "❌ HALLUCINATED"}[score]
    print(f"Score:   {label}")
    print(f"Comment: {result['comment']}")
    print()
    print("--- Response ---")
    print(response)

In [ ]:
print("=== BEFORE (no grounding instruction) ===")
retrieved_before, response_before = get_tool_outputs(fraud_claim, agent_before)
result_before = call_judge(retrieved_before, response_before)
show_result(result_before, response_before)

## 5 · Fix the prompt & re-evaluate

One line added to the prompt: *"Your rationale must QUOTE the exact text returned by the tools."*

### What changed?

```diff
  Investigate the claim and provide: ...

+ Always start by reading the policy to understand which checks are required.
+ Let the policy clauses guide which other tools you call.
+
+ Your rationale MUST quote exact text from the tool outputs —
+ do not assert facts not present in the retrieved content.
```

One instruction added. Let's see if it moves the grounding score.

In [ ]:
PROMPT_AFTER = """You are a claims investigation assistant for EuroShield Insurance Group.

You have access to four tools:
- search_policy_docs: retrieves the insurance policy
- query_claims_history: retrieves prior claims for a claimant
- query_weather_data: retrieves historical weather for a date and location
- retrieve_repair_estimate: retrieves contractor repair estimate for a claim

Always start by reading the policy to understand which checks are required.
Let the policy clauses guide which other tools you call.

Investigate the claim and provide:
- coverage_decision (covered / partial / excluded)
- settlement_recommendation (auto_settle / assign_adjuster / flag_for_investigation)
- confidence (high / medium / low)
- rationale explaining your decision

Your rationale MUST quote exact text from the tool outputs —
do not assert facts not present in the retrieved content."""

agent_after = create_agent(llm, tools, system_prompt=PROMPT_AFTER)

print("=== AFTER (grounding instruction added) ===")
retrieved_after, response_after = get_tool_outputs(fraud_claim, agent_after)
result_after = call_judge(retrieved_after, response_after)
show_result(result_after, response_after)

## 6 · Run across all claims with LangSmith evaluate()

Scale the same evaluator across the full dataset and compare experiments in LangSmith.

In [ ]:
from langsmith import Client
from langsmith.evaluation import evaluate

DATASET_NAME = "claims-investigation-workshop"
client = Client()

# Push dataset once — safe to re-run, skips if already exists
if DATASET_NAME not in {d.name for d in client.list_datasets()}:
    ds = client.create_dataset(dataset_name=DATASET_NAME)
    for claim in CLAIM_INPUTS:
        client.create_example(inputs={"claim": claim}, dataset_id=ds.id)
    print(f"Created dataset with {len(CLAIM_INPUTS)} examples.")
else:
    print(f"Dataset '{DATASET_NAME}' already exists.")

In [ ]:
# ← swap active_agent and prompt_version to run the other experiment
active_agent   = agent_after   # or agent_before
prompt_version = "after"       # or "before"

def target(inputs: dict) -> dict:
    claim_text = "\n".join(f"{k}: {v}" for k, v in inputs["claim"].items())
    state = active_agent.invoke({"messages": [HumanMessage(content=claim_text)]})
    return {"messages": state["messages"]}

def grounding_evaluator(run, example):
    tool_outputs = {}
    for msg in run.outputs.get("messages", []):
        if getattr(msg, "type", None) == "tool":
            tool_outputs[msg.name] = str(msg.content)
    retrieved = "\n\n".join(f"### {k}\n{v}" for k, v in tool_outputs.items()) or "No tool outputs."
    response  = run.outputs["messages"][-1].content
    result    = call_judge(retrieved, response)
    score     = _score(result["score"])
    return {"key": "grounding", "score": score, "comment": result["comment"]}

results = evaluate(
    target,
    data=DATASET_NAME,
    evaluators=[grounding_evaluator],
    experiment_prefix="grounding",
    metadata={"prompt_version": prompt_version},
)

scores = [r["evaluation_results"]["results"][0].score for r in results]
labels = {1.0: "✅ GROUNDED", 0.5: "⚠️  PARTIAL", 0.0: "❌ HALLUCINATED"}
for claim, score in zip(CLAIM_INPUTS, scores):
    print(f"  {claim['claim_id']}  →  {labels.get(score, score)}")
print(f"\nMean grounding score: {sum(scores)/len(scores):.2f}")

print(f"\nView experiment comparison in LangSmith:")
print(f"https://smith.langchain.com/")
print(f"→ Open project 'claims-workshop' → Datasets → {DATASET_NAME} → Compare experiments")